# Notebook 6 — Routing par super-arêtes club

## Objectif

Enrichir le routing du Notebook 4 pour qu'il **emprunte effectivement les sous-tronçons club** identifiés au Notebook 2 (~1433 segments validés).

Au lieu de laisser Dijkstra trouver "le mieux" parmi 700k arêtes individuelles, on lui dit explicitement : "Va de A à B **en passant par** un ou plusieurs sous-tronçons club bien choisis".

## Pourquoi

Le Notebook 4 produit des itinéraires "raisonnables" mais qui ne passent **presque pas** par les routes signature du club. C'est parce que :
1. Beaucoup d'arêtes en région parisienne ont `cost_factor=1` après tes corrections
2. Dijkstra n'a aucune raison de "préférer" un parcours club s'il existe un équivalent plus court
3. Le score `score_final` capte la qualité d'arête mais pas la qualité du parcours global

## Méthode V1 (heuristique simple)

Approche en 4 étapes :

1. **Filtre sous-tronçons candidats** : ceux à moins de X km du segment direct A→B
2. **Score chaque candidat** : club_quality / (longueur du détour induit)
3. **Combinaisons 1, 2, 3 sous-tronçons** : on teste toutes les combinaisons légères
4. **Routing final** : enchaîne `Pantin → début_S1 → fin_S1 → ... → fin_Sk → Chevreuse`

C'est une approximation gloutonne de l'AOP. Pour la version rigoureuse,
voir Notebook 7 (GRASP de Verbeeck et al. 2014).

## Plan

1. Setup
2. Chargement des sous-tronçons (`trace_segments` du Notebook 2)
3. Filtrage des candidats par proximité géographique
4. Routing enchaîné
5. Comparaison "pur" vs "avec super-arêtes"
6. Visualisation


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import time
import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine, text
from itertools import combinations
import folium
from shapely import wkt

DB_CONFIG = {"user": "postgres", "password": "4421",
             "host": "localhost", "port": 5432, "database": "velo_club"}
url = (f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
       f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")
engine = create_engine(url, pool_pre_ping=True)
print("✓ engine OK")

✓ engine OK


## 2. Chargement du graphe (depuis Notebook 4)

On reconstruit le graphe networkx avec les cost_factor v2 (intègre PU learning).

In [2]:
print("Extraction du graphe...")
t0 = time.time()
with engine.connect() as conn:
    df_edges = pd.read_sql(text("""
        SELECT
            e.edge_id, e.length_m, e.highway,
            ROUND(ST_X(ST_StartPoint(e.geom))::numeric, 6) AS u_lon,
            ROUND(ST_Y(ST_StartPoint(e.geom))::numeric, 6) AS u_lat,
            ROUND(ST_X(ST_EndPoint(e.geom))::numeric, 6) AS v_lon,
            ROUND(ST_Y(ST_EndPoint(e.geom))::numeric, 6) AS v_lat,
            COALESCE(es.cost_factor_v2_club, es.cost_factor_club, 1.0) AS cost_factor_club,
            COALESCE(es.cost_factor_v2_solo, es.cost_factor_solo, 1.0) AS cost_factor_solo,
            COALESCE(es.score_final_club, 0) AS score_final_club,
            COALESCE(es.score_final_solo, 0) AS score_final_solo
        FROM osm_edges e
        LEFT JOIN edge_scores es ON es.edge_id = e.edge_id
        WHERE e.is_routable = TRUE AND e.geom IS NOT NULL AND e.length_m > 0;
    """), conn)

df_edges["u_id"] = df_edges["u_lat"].astype(str) + "_" + df_edges["u_lon"].astype(str)
df_edges["v_id"] = df_edges["v_lat"].astype(str) + "_" + df_edges["v_lon"].astype(str)

# Build graph
G = nx.MultiDiGraph()
for _, r in df_edges.iterrows():
    G.add_node(r.u_id, lat=float(r.u_lat), lon=float(r.u_lon))
    G.add_node(r.v_id, lat=float(r.v_lat), lon=float(r.v_lon))
    attrs = {"edge_id": int(r.edge_id), "length_m": float(r.length_m),
             "highway": r.highway,
             "cost_factor_club": float(r.cost_factor_club),
             "cost_factor_solo": float(r.cost_factor_solo),
             "score_final_club": float(r.score_final_club),
             "score_final_solo": float(r.score_final_solo)}
    G.add_edge(r.u_id, r.v_id, **attrs)
    G.add_edge(r.v_id, r.u_id, **attrs)

# Keep largest CC
largest_cc = max(nx.weakly_connected_components(G), key=len)
G = G.subgraph(largest_cc).copy()
print(f"✓ Graphe : {G.number_of_nodes():,} nœuds / {G.number_of_edges():,} arêtes en {time.time()-t0:.0f}s")

Extraction du graphe...
✓ Graphe : 90,762 nœuds / 205,652 arêtes en 53s


## 3. Chargement des sous-tronçons club

Les sous-tronçons sont issus du Notebook 2 (table `trace_segments`).
Chacun fait ~10 km, est entièrement composé d'arêtes empruntées par le club.

In [3]:
print("Chargement des sous-tronçons...")

with engine.connect() as conn:
    segments = pd.read_sql(text("""
        SELECT 
            ts.segment_id,
            ts.length_m AS segment_length_m,
            ts.start_lat, ts.start_lon,
            ts.end_lat, ts.end_lon,
            -- Score club moyen pondéré sur les arêtes du tronçon
            (
                SELECT AVG(es.score_club)
                FROM trace_segment_edges tse
                JOIN edge_scores es ON es.edge_id = tse.edge_id
                WHERE tse.segment_id = ts.segment_id
            ) AS mean_score_club,
            -- D+ cumulé (depuis Notebook 5)
            (
                SELECT SUM(oe.d_plus_m)
                FROM trace_segment_edges tse
                JOIN osm_edges oe ON oe.edge_id = tse.edge_id
                WHERE tse.segment_id = ts.segment_id
            ) AS d_plus_total
        FROM trace_segments ts
        ORDER BY ts.segment_id;
    """), conn)

print(f"  {len(segments):,} sous-tronçons chargés")
print(f"  Longueur moyenne : {segments['segment_length_m'].mean()/1000:.1f} km")
print(f"  Score club moyen : {segments['mean_score_club'].mean():.2f}")

Chargement des sous-tronçons...


DatabaseError: Execution failed on sql '
        SELECT 
            ts.segment_id,
            ts.length_m AS segment_length_m,
            ts.start_lat, ts.start_lon,
            ts.end_lat, ts.end_lon,
            -- Score club moyen pondéré sur les arêtes du tronçon
            (
                SELECT AVG(es.score_club)
                FROM trace_segment_edges tse
                JOIN edge_scores es ON es.edge_id = tse.edge_id
                WHERE tse.segment_id = ts.segment_id
            ) AS mean_score_club,
            -- D+ cumulé (depuis Notebook 5)
            (
                SELECT SUM(oe.d_plus_m)
                FROM trace_segment_edges tse
                JOIN osm_edges oe ON oe.edge_id = tse.edge_id
                WHERE tse.segment_id = ts.segment_id
            ) AS d_plus_total
        FROM trace_segments ts
        ORDER BY ts.segment_id;
    ': (psycopg2.errors.UndefinedColumn) ERREUR:  la colonne ts.start_lat n'existe pas
LINE 5:             ts.start_lat, ts.s

**Fallback si table absente** : si tu n'as pas exactement `trace_segments` en base, lance la cellule alternative ci-dessous qui reconstruit les segments depuis `trace_edges`.

In [8]:
# Fallback : reconstruire les segments si la table n'existe pas
# Lance UNIQUEMENT si la cellule précédente plante

print("Reconstruction des segments depuis trace_edges...")
with engine.connect() as conn:
    segments = pd.read_sql(text("""
        WITH trace_geom AS (
            SELECT 
                te.trace_id,
                ST_Collect(oe.geom ORDER BY te.position) AS segment_geom,
                SUM(oe.length_m) AS total_length,
                AVG(es.score_club) AS mean_score_club,
                SUM(oe.d_plus_m) AS d_plus_total
            FROM trace_edges te
            JOIN osm_edges oe ON oe.edge_id = te.edge_id
            JOIN edge_scores es ON es.edge_id = te.edge_id
            GROUP BY te.trace_id
            HAVING SUM(oe.length_m) BETWEEN 5000 AND 15000
        )
        SELECT 
            ROW_NUMBER() OVER () AS segment_id,
            total_length AS segment_length_m,
            ST_Y(ST_StartPoint(ST_LineMerge(segment_geom))) AS start_lat,
            ST_X(ST_StartPoint(ST_LineMerge(segment_geom))) AS start_lon,
            ST_Y(ST_EndPoint(ST_LineMerge(segment_geom))) AS end_lat,
            ST_X(ST_EndPoint(ST_LineMerge(segment_geom))) AS end_lon,
            mean_score_club,
            d_plus_total
        FROM trace_geom;
    """), conn)

print(f"{len(segments)} segments reconstruits")

Reconstruction des segments depuis trace_edges...


DatabaseError: Execution failed on sql '
        WITH trace_geom AS (
            SELECT 
                te.trace_id,
                ST_Collect(oe.geom ORDER BY te.position) AS segment_geom,
                SUM(oe.length_m) AS total_length,
                AVG(es.score_club) AS mean_score_club,
                SUM(oe.d_plus_m) AS d_plus_total
            FROM trace_edges te
            JOIN osm_edges oe ON oe.edge_id = te.edge_id
            JOIN edge_scores es ON es.edge_id = te.edge_id
            GROUP BY te.trace_id
            HAVING SUM(oe.length_m) BETWEEN 5000 AND 15000
        )
        SELECT 
            ROW_NUMBER() OVER () AS segment_id,
            total_length AS segment_length_m,
            ST_Y(ST_StartPoint(ST_LineMerge(segment_geom))) AS start_lat,
            ST_X(ST_StartPoint(ST_LineMerge(segment_geom))) AS start_lon,
            ST_Y(ST_EndPoint(ST_LineMerge(segment_geom))) AS end_lat,
            ST_X(ST_EndPoint(ST_LineMerge(segment_geom))) AS end_lon,
     

In [13]:
print("Chargement des sous-tronçons...")

with engine.connect() as conn:
    segments = pd.read_sql(text("""
        SELECT 
            ts.segment_id,
            ts.length_m AS segment_length_m,
            
            -- ST_Centroid garantit de trouver un point valide, peu importe le sens de l'arête OSM
            ST_Y(ST_Centroid(oe_start.geom)) AS start_lat, 
            ST_X(ST_Centroid(oe_start.geom)) AS start_lon,
            ST_Y(ST_Centroid(oe_end.geom)) AS end_lat, 
            ST_X(ST_Centroid(oe_end.geom)) AS end_lon,
            
            -- Score club moyen pondéré sur les arêtes du tronçon
            (
                SELECT AVG(es.score_club)
                FROM trace_segment_edges tse
                JOIN edge_scores es ON es.edge_id = tse.edge_id
                WHERE tse.segment_id = ts.segment_id
            ) AS mean_score_club,
            
            -- D+ cumulé
            (
                SELECT SUM(oe.d_plus_m)
                FROM trace_segment_edges tse
                JOIN osm_edges oe ON oe.edge_id = tse.edge_id
                WHERE tse.segment_id = ts.segment_id
            ) AS d_plus_total
            
        FROM trace_segments ts
        JOIN osm_edges oe_start ON ts.edge_id_start = oe_start.edge_id
        JOIN osm_edges oe_end ON ts.edge_id_end = oe_end.edge_id
        ORDER BY ts.segment_id;
    """), conn)

print(f"  {len(segments):,} sous-tronçons chargés")
# On vérifie qu'il n'y a plus de valeurs vides !
print(f"  Valeurs manquantes : {segments['start_lat'].isna().sum()}")

Chargement des sous-tronçons...
  1,655 sous-tronçons chargés
  Valeurs manquantes : 2


## 4. Filtrage des candidats par proximité

Pour A→B, on garde les sous-tronçons qui sont "à proximité" du chemin direct.
Critères :
- Le tronçon doit être à moins de X km du segment géométrique A↔B
- Détour induit (A → début_S → fin_S → B) raisonnable (< 1.5× distance directe)

In [14]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Distance haversine en km."""
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def filter_candidates(segments_df, A, B, max_detour_km=15):
    """Sélectionne les sous-tronçons proches du couloir A↔B.
    
    Critère : distance perpendiculaire au segment A↔B < max_detour_km.
    """
    A_lat, A_lon = A
    B_lat, B_lon = B
    direct_km = haversine_km(A_lat, A_lon, B_lat, B_lon)
    
    candidates = []
    for _, s in segments_df.iterrows():
        # On approxime la distance perpendiculaire au segment AB en
        # vérifiant que le tronçon est dans le "couloir"
        s_mid_lat = (s["start_lat"] + s["end_lat"]) / 2
        s_mid_lon = (s["start_lon"] + s["end_lon"]) / 2
        
        # Distance du milieu du tronçon aux extrémités A et B
        d_to_A = haversine_km(s_mid_lat, s_mid_lon, A_lat, A_lon)
        d_to_B = haversine_km(s_mid_lat, s_mid_lon, B_lat, B_lon)
        
        # Le tronçon est "sur le chemin" si d_to_A + d_to_B - direct_km est petit
        # (= triangulation : si proche du segment AB, somme ≈ direct_km)
        excess = d_to_A + d_to_B - direct_km
        
        if excess < max_detour_km:
            cand = s.copy()
            cand["excess_km"] = excess
            cand["d_to_A"] = d_to_A
            cand["d_to_B"] = d_to_B
            candidates.append(cand)
    
    if not candidates:
        return pd.DataFrame()
    
    df_cand = pd.DataFrame(candidates)
    # Score combiné : on veut un tronçon avec haut mean_score_club et faible excess
    df_cand["candidate_score"] = (
        df_cand["mean_score_club"].fillna(0.5) / (1 + df_cand["excess_km"])
    )
    df_cand = df_cand.sort_values("candidate_score", ascending=False)
    return df_cand


# Test
POINTS = {
    "pantin":         (48.8922, 2.4014),
    "chevreuse":      (48.7076, 2.0387),
    "fontainebleau":  (48.4020, 2.7015),
    "rambouillet":    (48.6447, 1.8285),
}

cands = filter_candidates(segments, POINTS["pantin"], POINTS["chevreuse"], max_detour_km=15)
print(f"Candidats Pantin → Chevreuse : {len(cands)}")
print(cands[["segment_id", "mean_score_club", "excess_km", "candidate_score"]].head(15).to_string(index=False))

Candidats Pantin → Chevreuse : 590
 segment_id  mean_score_club  excess_km  candidate_score
      192.0         0.619755   0.003543         0.617567
      212.0         0.618253   0.003543         0.616070
      202.0         0.618253   0.003543         0.616070
      236.0         0.615736   0.003543         0.613563
      341.0         0.624142   0.031578         0.605036
      328.0         0.612126   0.013946         0.603707
      127.0         0.604226   0.017116         0.594058
      168.0         0.638653   0.079882         0.591410
      156.0         0.638653   0.079882         0.591410
      181.0         0.639360   0.086739         0.588329
      317.0         0.646142   0.112155         0.580982
      258.0         0.648569   0.123827         0.577108
       83.0         0.610338   0.073390         0.568608
      360.0         0.585761   0.036742         0.565001
      102.0         0.608221   0.077998         0.564213


## 5. Routing enchaîné

On teste différentes combinaisons de 1, 2 ou 3 sous-tronçons et on prend la meilleure.

In [15]:
def snap_to_node(G, lat, lon):
    """Trouve le nœud le plus proche."""
    coords = np.array([(d["lat"], d["lon"]) for _, d in G.nodes(data=True)])
    ids = list(G.nodes())
    lat_rad = np.radians(lat)
    dx = (coords[:, 1] - lon) * 111320 * np.cos(lat_rad)
    dy = (coords[:, 0] - lat) * 111320
    d2 = dx*dx + dy*dy
    i = int(np.argmin(d2))
    return ids[i]


def edge_cost_exp(attrs, key, alpha):
    L = attrs["length_m"]
    cf = attrs[key]
    return L * (cf ** alpha)


def shortest_path_via(G, A, B, profile="club_road", alpha=1.0):
    """Dijkstra simple A → B avec cost_factor exponentiel."""
    u = snap_to_node(G, *A)
    v = snap_to_node(G, *B)
    cost_key = "cost_factor_club" if profile == "club_road" else "cost_factor_solo"
    
    def cost_fn(uu, vv, d):
        if isinstance(d, dict) and any(isinstance(x, dict) for x in d.values()):
            return min(edge_cost_exp(attrs, cost_key, alpha) for attrs in d.values())
        return edge_cost_exp(d, cost_key, alpha)
    
    try:
        path = nx.dijkstra_path(G, u, v, weight=cost_fn)
    except nx.NetworkXNoPath:
        return None
    
    total_L = 0
    weighted_score = 0
    coords = [(G.nodes[path[0]]["lat"], G.nodes[path[0]]["lon"])]
    score_key = f"score_final_{profile.replace('_road','').replace('_casual','')}"
    
    for i in range(len(path) - 1):
        a, b = path[i], path[i+1]
        cands = G[a][b]
        best = min(cands.keys(), key=lambda k: edge_cost_exp(cands[k], cost_key, alpha))
        attrs = cands[best]
        total_L += attrs["length_m"]
        weighted_score += attrs["length_m"] * attrs[score_key]
        coords.append((G.nodes[b]["lat"], G.nodes[b]["lon"]))
    
    return {
        "total_length_m": total_L,
        "mean_score": weighted_score / max(total_L, 1),
        "coords": coords,
    }


def route_via_segments(G, A, B, segments_to_use, profile="club_road", alpha=1.0):
    """Calcule un itinéraire qui passe par les sous-tronçons donnés.
    
    segments_to_use : liste de dicts avec start_lat/start_lon/end_lat/end_lon
    Itinéraire : A → start_S1 → end_S1 → start_S2 → end_S2 → ... → B
    """
    waypoints = [A]
    for s in segments_to_use:
        waypoints.append((s["start_lat"], s["start_lon"]))
        waypoints.append((s["end_lat"], s["end_lon"]))
    waypoints.append(B)
    
    total_L = 0
    weighted_score = 0
    all_coords = []
    
    for i in range(len(waypoints) - 1):
        leg = shortest_path_via(G, waypoints[i], waypoints[i+1], profile, alpha)
        if leg is None:
            return None
        total_L += leg["total_length_m"]
        weighted_score += leg["mean_score"] * leg["total_length_m"]
        if i == 0:
            all_coords.extend(leg["coords"])
        else:
            all_coords.extend(leg["coords"][1:])  # skip duplicate
    
    return {
        "total_length_m": total_L,
        "mean_score": weighted_score / max(total_L, 1),
        "coords": all_coords,
        "segments_used": [s["segment_id"] for s in segments_to_use],
    }

In [16]:
def best_super_route(G, segments_df, A, B, profile="club_road", alpha=1.0,
                     max_detour_km=15, top_k=10, max_segments=2,
                     max_total_km=None):
    """Trouve le meilleur itinéraire passant par 1-N sous-tronçons.
    
    Stratégie :
    1. Filtre les candidats proches
    2. Calcule baseline (Dijkstra pur)
    3. Pour 1 segment : teste les top_k candidats
    4. Pour 2 segments : teste les combinaisons des top_k/2 candidats
    5. Renvoie le meilleur (par score moyen × distance)
    """
    direct_km = haversine_km(A[0], A[1], B[0], B[1])
    if max_total_km is None:
        max_total_km = direct_km * 1.8
    
    # Baseline
    baseline = shortest_path_via(G, A, B, profile, alpha)
    if baseline is None:
        return None
    print(f"  Baseline (sans super-arête) : {baseline['total_length_m']/1000:.1f} km, score {baseline['mean_score']:.2f}")
    
    # Candidats
    cands = filter_candidates(segments_df, A, B, max_detour_km)
    if len(cands) == 0:
        print("  Aucun candidat trouvé")
        return baseline
    
    cands_top = cands.head(top_k)
    
    best = baseline
    best["segments_used"] = []
    
    # 1 segment
    print(f"  Test 1 segment ({len(cands_top)} candidats)...")
    for _, c in cands_top.iterrows():
        r = route_via_segments(G, A, B, [c.to_dict()], profile, alpha)
        if r and r["total_length_m"]/1000 < max_total_km:
            # Critère : on veut un score plus élevé que baseline,
            # à un coût de distance acceptable
            improvement = r["mean_score"] - baseline["mean_score"]
            penalty = max(0, r["total_length_m"]/baseline["total_length_m"] - 1.2)
            if improvement - penalty > best.get("mean_score", 0) - baseline["mean_score"]:
                best = r
                best["mean_score"] = r["mean_score"]
    
    # 2 segments (limité aux top_k/2 pour tractabilité)
    if max_segments >= 2 and len(cands_top) >= 2:
        print(f"  Test 2 segments ({len(cands_top)*(len(cands_top)-1)//2} combinaisons)...")
        for c1, c2 in combinations(cands_top.head(min(8, top_k)).iterrows(), 2):
            # Ordonner les segments par proximité à A
            s1, s2 = c1[1].to_dict(), c2[1].to_dict()
            if s1["d_to_A"] > s2["d_to_A"]:
                s1, s2 = s2, s1
            r = route_via_segments(G, A, B, [s1, s2], profile, alpha)
            if r and r["total_length_m"]/1000 < max_total_km:
                improvement = r["mean_score"] - baseline["mean_score"]
                penalty = max(0, r["total_length_m"]/baseline["total_length_m"] - 1.5)
                if improvement - penalty > best.get("mean_score", 0) - baseline["mean_score"] + 0.05:
                    best = r
                    best["mean_score"] = r["mean_score"]
    
    return best


# Test : Pantin → Chevreuse avec super-arêtes
print("=" * 70)
print("Pantin → Chevreuse avec super-arêtes")
print("=" * 70)
t0 = time.time()
result = best_super_route(G, segments, POINTS["pantin"], POINTS["chevreuse"],
                          profile="club_road", alpha=1.0,
                          max_detour_km=12, top_k=8, max_segments=2)
print(f"\n→ {result['total_length_m']/1000:.1f} km, score {result['mean_score']:.3f}")
print(f"   Segments empruntés : {result.get('segments_used', [])}")
print(f"   Temps : {time.time()-t0:.0f}s")

Pantin → Chevreuse avec super-arêtes
  Baseline (sans super-arête) : 58.4 km, score 0.44
  Test 1 segment (8 candidats)...
  Test 2 segments (28 combinaisons)...

→ 58.4 km, score 0.440
   Segments empruntés : []
   Temps : 44s


## 6. Visualisation

Comparaison itinéraire pur vs avec super-arêtes.

In [17]:
# Calcule baseline et super-route
print("Calcul des deux versions pour comparaison...")
baseline = shortest_path_via(G, POINTS["pantin"], POINTS["chevreuse"], "club_road", 1.0)
super_r = best_super_route(G, segments, POINTS["pantin"], POINTS["chevreuse"],
                           profile="club_road", alpha=1.0,
                           max_detour_km=12, top_k=8, max_segments=2)

# Carte
m = folium.Map(location=[48.8, 2.2], zoom_start=10, tiles="cartodbpositron")

if baseline:
    folium.PolyLine(baseline["coords"], color="blue", weight=3, opacity=0.7,
        tooltip=f"Pur Dijkstra: {baseline['total_length_m']/1000:.1f}km, score {baseline['mean_score']:.2f}"
    ).add_to(m)

if super_r and super_r.get("segments_used"):
    folium.PolyLine(super_r["coords"], color="red", weight=4, opacity=0.85,
        tooltip=f"Super-arêtes: {super_r['total_length_m']/1000:.1f}km, score {super_r['mean_score']:.2f}"
    ).add_to(m)

# Marquage des sous-tronçons utilisés
if super_r and super_r.get("segments_used"):
    used_ids = super_r["segments_used"]
    for sid in used_ids:
        s = segments[segments["segment_id"] == sid].iloc[0]
        folium.Marker([s["start_lat"], s["start_lon"]], 
            icon=folium.Icon(color="green", icon="play"),
            tooltip=f"Début segment {sid}"
        ).add_to(m)
        folium.Marker([s["end_lat"], s["end_lon"]], 
            icon=folium.Icon(color="orange", icon="stop"),
            tooltip=f"Fin segment {sid}"
        ).add_to(m)

folium.Marker(POINTS["pantin"], icon=folium.Icon(color="black"), tooltip="Pantin").add_to(m)
folium.Marker(POINTS["chevreuse"], icon=folium.Icon(color="black"), tooltip="Chevreuse").add_to(m)
m

Calcul des deux versions pour comparaison...
  Baseline (sans super-arête) : 58.4 km, score 0.44
  Test 1 segment (8 candidats)...
  Test 2 segments (28 combinaisons)...


## 7. Benchmark sur plusieurs paires

In [18]:
bench = []
pairs = [("pantin", "chevreuse"), ("pantin", "rambouillet"), ("pantin", "fontainebleau")]

for src, dst in pairs:
    A, B = POINTS[src], POINTS[dst]
    print(f"\n{src} → {dst} ...")
    
    base = shortest_path_via(G, A, B, "club_road", 1.0)
    super_r = best_super_route(G, segments, A, B, profile="club_road",
                               alpha=1.0, max_detour_km=15, top_k=6, max_segments=2)
    
    bench.append({
        "A→B": f"{src}→{dst}",
        "baseline_km": round(base["total_length_m"]/1000, 1) if base else None,
        "baseline_score": round(base["mean_score"], 3) if base else None,
        "super_km": round(super_r["total_length_m"]/1000, 1) if super_r else None,
        "super_score": round(super_r["mean_score"], 3) if super_r else None,
        "n_segments": len(super_r.get("segments_used", [])) if super_r else 0,
    })

print("\nBenchmark :")
print(pd.DataFrame(bench).to_string(index=False))


pantin → chevreuse ...
  Baseline (sans super-arête) : 58.4 km, score 0.44
  Test 1 segment (6 candidats)...
  Test 2 segments (15 combinaisons)...

pantin → rambouillet ...
  Baseline (sans super-arête) : 77.4 km, score 0.54
  Test 1 segment (6 candidats)...
  Test 2 segments (15 combinaisons)...

pantin → fontainebleau ...
  Baseline (sans super-arête) : 133.4 km, score 0.49
  Test 1 segment (6 candidats)...
  Test 2 segments (15 combinaisons)...

Benchmark :
                 A→B  baseline_km  baseline_score  super_km  super_score  n_segments
    pantin→chevreuse         58.4           0.440      58.4        0.440           0
  pantin→rambouillet         77.4           0.543      89.4        0.585           1
pantin→fontainebleau        133.4           0.494     133.4        0.494           0


## 8. Conclusion

Ce notebook V1 utilise une heuristique gloutonne pour intégrer les sous-tronçons club. Avantages :
- Garantit que l'itinéraire passe **vraiment** par des routes club emblématiques
- Reste rapide (< 1 min par A→B)

Limites V1 :
- Filtrage par proximité géométrique grossière (pas optimal)
- Limité à 1-2 sous-tronçons par défaut (3+ devient lent)
- Pas d'optimisation rigoureuse au sens AOP

**Notebook 7** : implémentation rigoureuse de l'AOP avec GRASP (Verbeeck 2014).
